# Exercise 3 — calculate_price and generate_pricing_tiers

Pricing an AI product starts with knowing your costs. `calculate_price` uses cost-plus pricing: you know what each API call costs you (LLM tokens, compute), you decide your target margin, and the formula tells you what to charge. `generate_pricing_tiers` applies this to multiple tiers with different call volumes and margins.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class ProductConfig:
    name: str; version: str; description: str
    author: str; email: str
    dependencies: list = field(default_factory=list)
    license: str  = "MIT"
    python_requires: str = ">=3.10"

_CFG = ProductConfig(
    name="my-trading-bot",
    version="0.1.0",
    description="AI-powered paper-trading bot using sentiment and technical signals.",
    author="Jane Doe",
    email="jane@example.com",
    dependencies=["pandas>=2.0", "requests>=2.28"],
)

def calculate_price(cost_per_call, calls_per_month, margin=0.5):
    """Cost-plus pricing: price = total_cost / (1 − margin).

    Example: cost=$0.001/call, 1000 calls, margin=0.5
      total_cost = $1.00
      price = $1.00 / 0.5 = $2.00

    Raises ValueError if margin not in [0, 1).
    Returns float rounded to 2 decimal places.
    """
    # TODO: ~4 lines
    return 0.0


def generate_pricing_tiers(cost_per_call, tiers):
    """Generate SaaS pricing tiers.

    Args:
        cost_per_call : float
        tiers : list[dict] with keys:
          "name"   : str
          "calls"  : int
          "margin" : float (optional, default 0.5)

    Returns:
        list[dict] with keys: name, calls, price_per_month, price_per_call
    """
    # TODO: ~7 lines
    return []


### Checks

In [ ]:
checks = 0

# 1 — calculate_price: known example
try:
    p = calculate_price(0.001, 1000, margin=0.5)
    assert abs(p - 2.0) < 1e-9, f"expected $2.00, got ${p}"
    checks += 1; print("✅ 1 calculate_price(0.001, 1000, 0.5) = $2.00")
except Exception as e:
    print("❌ 1:", e)

# 2 — margin=0.8 → 5× cost
try:
    p = calculate_price(0.001, 1000, margin=0.8)
    assert abs(p - 5.0) < 1e-9, f"expected $5.00 (80% margin), got ${p}"
    checks += 1; print("✅ 2 calculate_price(0.001, 1000, 0.8) = $5.00 (80% margin)")
except Exception as e:
    print("❌ 2:", e)

# 3 — invalid margin raises ValueError
try:
    try:
        calculate_price(0.001, 1000, margin=1.0)
        print("❌ 3: expected ValueError for margin=1.0")
    except ValueError:
        checks += 1; print("✅ 3 margin=1.0 raises ValueError")
except Exception as e:
    print("❌ 3:", e)

# 4 — generate_pricing_tiers: structure
try:
    tiers = [
        {"name": "Free",  "calls":    100},
        {"name": "Indie", "calls":  1_000},
        {"name": "Pro",   "calls": 10_000, "margin": 0.7},
    ]
    result = generate_pricing_tiers(0.001, tiers)
    assert len(result) == 3
    for t in result:
        assert {"name","calls","price_per_month","price_per_call"}.issubset(t.keys())
    checks += 1; print("✅ 4 generate_pricing_tiers returns 3 dicts with correct keys")
except Exception as e:
    print("❌ 4:", e)

# 5 — pricing math is correct across tiers
try:
    tiers = [
        {"name": "Free",  "calls":   100, "margin": 0.5},
        {"name": "Pro",   "calls": 1_000, "margin": 0.5},
    ]
    result = generate_pricing_tiers(0.001, tiers)
    # Free: 0.001*100 / 0.5 = 0.20
    assert abs(result[0]["price_per_month"] - 0.20) < 1e-6,         f"Free tier: expected $0.20, got ${result[0]['price_per_month']}"
    # Pro: 0.001*1000 / 0.5 = 2.00
    assert abs(result[1]["price_per_month"] - 2.00) < 1e-6,         f"Pro tier: expected $2.00, got ${result[1]['price_per_month']}"
    # price_per_call = price_per_month / calls
    assert abs(result[0]["price_per_call"] - 0.002) < 1e-6
    checks += 1; print("✅ 5 pricing math correct: Free=$0.20, Pro=$2.00")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
